# QPOP's quadratic vs DECREASE vs Bliss, same measurements

Same 192 measured 8x8 matrices as `real_surfaces_check.ipynb`. Every method sees the **same 22 points**: both single-drug curves (15 points) and the diagonal (7 points),
the low-cost design the DECREASE paper describes. Each method then predicts the 42 combinations that were not measured, which are compared with the real measurements.

- **Quadratic (QPOP's model):** 6-term second-order surface in concentration index, least squares on the 22 points.
- **DECREASE:** the published example model (`DECREASE_model_example/DECREASEmodel.R`, cNMF + XGBoost ensemble), run unmodified by `scripts/run_decrease.R`
  with R 4.6.1 and xgboost 1.7.11.1 (the model uses the pre-3.0 xgboost interface). Predictions are stored in `ext/decrease_runs/`.
- **Bliss:** independent action from the two single-drug curves only (15 points; ignores the diagonal).

Run order: `sh scripts/fetch_open_data.sh`, then `Rscript scripts/run_decrease.R <ids.txt> <out.csv>` (about 50-80 s per matrix). Here: a random subset of 60 of the 192 matrices (numpy seed 2026), for run time.

In [1]:
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go

raw = pd.read_excel("../ext/decrease/210_Novel_Anticancer_combinations/192_Combinations_.xlsx")
dec = pd.read_csv("../data/decrease_predictions_60_matrices.csv")   # output of scripts/run_decrease.R
print("DECREASE predictions for", dec.PairIndex.nunique(), "matrices; median seconds per matrix:", round(dec.groupby("PairIndex").seconds.first().median()))

I, J = np.meshgrid(np.arange(8), np.arange(8), indexing="ij")
def F(i, j):
    i, j = np.asarray(i, float), np.asarray(j, float)
    return np.stack([np.ones_like(i), i, j, i * j, i * i, j * j], -1)
MEAS = (I == 0) | (J == 0) | (I == J)          # single-agent row/column + diagonal: 22 points
TEST = ~MEAS                                   # 42 unmeasured combinations

rows = []
for k, g in raw.groupby("PairIndex"):
    d = dec[dec.PairIndex == k]
    if d.empty:
        continue
    c1, c2 = np.sort(g.Conc1.unique()), np.sort(g.Conc2.unique())
    V = g.pivot_table(index="Conc1", columns="Conc2", values="Response").loc[c1, c2].to_numpy()
    P_dec = d.pivot_table(index="Conc1", columns="Conc2", values="pred").loc[c1, c2].to_numpy()
    b = np.linalg.lstsq(F(I[MEAS], J[MEAS]), V[MEAS], rcond=None)[0]
    P_quad = (F(I.ravel(), J.ravel()) @ b).reshape(8, 8)
    P_bliss = V[0, 0] * np.outer(V[:, 0] / V[0, 0], V[0, :] / V[0, 0])
    r = dict(k=k, cell=g.Cell.iloc[0], pair=f"{g.Drug1.iloc[0]} + {g.Drug2.iloc[0]}")
    best = V[TEST].min()
    for name, P in (("quadratic", P_quad), ("DECREASE", P_dec), ("Bliss", P_bliss)):
        r[f"rmse_{name}"] = float(np.sqrt(((P - V)[TEST] ** 2).mean()))
        r[f"regret_{name}"] = float(V[TEST][P[TEST].argmin()] - best)
    rows.append(r)
R = pd.DataFrame(rows)
summary = pd.DataFrame({m: [R[f"rmse_{m}"].median(), R[f"rmse_{m}"].mean(), R[f"regret_{m}"].mean()]
                        for m in ("quadratic", "DECREASE", "Bliss")}, index=["median RMSE", "mean RMSE", "mean regret"]).round(2)
print(summary.to_string())
print("units: viability percentage points")
for a, b2 in (("DECREASE", "quadratic"), ("Bliss", "quadratic"), ("DECREASE", "Bliss")):
    print(f"{a} more accurate than {b2} on {(R[f'rmse_{a}'] < R[f'rmse_{b2}']).mean():.0%} of matrices")

DECREASE predictions for 60 matrices; median seconds per matrix: 95


             quadratic  DECREASE  Bliss
median RMSE      10.46      5.51   7.89
mean RMSE        11.07      6.80   9.40
mean regret       2.90      2.48   3.41
units: viability percentage points
DECREASE more accurate than quadratic on 78% of matrices
Bliss more accurate than quadratic on 57% of matrices
DECREASE more accurate than Bliss on 92% of matrices


In [2]:
fig = go.Figure()
for m in ("quadratic", "DECREASE", "Bliss"):
    fig.add_trace(go.Box(y=R[f"rmse_{m}"], name=m, boxpoints="outliers"))
fig.update_layout(yaxis_title="RMSE on the 42 unmeasured combinations (viability %)", height=430,
                  title="Same 22 measurements, three ways to predict the rest")
fig

In [3]:
lim = float(R[["rmse_quadratic", "rmse_DECREASE"]].max().max()) * 1.05
fig = go.Figure(go.Scatter(x=R.rmse_DECREASE, y=R.rmse_quadratic, mode="markers", opacity=0.6, text=R.pair + " (" + R.cell + ")"))
fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode="lines", line=dict(dash="dot")))
fig.update_layout(xaxis_title="DECREASE RMSE", yaxis_title="quadratic RMSE", showlegend=False, height=480,
                  title="Above the line: DECREASE was more accurate")
fig

## What this says

On 60 randomly chosen real matrices, with every method given the same 22 measurements:

1. **DECREASE predicts the unmeasured combinations about twice as accurately as QPOP's quadratic** (median RMSE 5.5 vs 10.5 viability points) and beats it on 78% of matrices.
2. **The quadratic is worse than the plain Bliss baseline** too (median 7.9), which needs only the two single-drug curves.
3. **More data does not rescue the quadratic.** Its error with 22 points (10.5) is about the same as with QPOP's 9-point grid in `real_surfaces_check.ipynb` (10.6): the shape is wrong, not just under-sampled.
4. **For picking the single best dose pair the methods are close** (mean regret 2.5, 2.9 and 3.4 points); the big differences are in mapping the whole surface, which is what QPOP's rankings rely on.

Limits: cell lines, one measurement per point, one subset of 60 of 192 matrices (seed 2026), and DECREASE is the published example model run once with its own random seed (set.seed(42) in `DECREASEmodel.R`).
Building on existing work this way, an open, published method already does QPOP's job of filling in a combination surface better than QPOP's own model.